# RAG Interaction Study — Full Experiment Suite
**Paper:** *How Chunking, Embedding, and Reranking Interact in Long-Document RAG*

**Runtime:** ~4 hours on free Colab T4 (16GB)

**Output:** results_quality.csv and results_narrativeqa.csv.

In [1]:
# Cell 1: Install dependencies
!pip install -q rank_bm25 sentence-transformers datasets nltk transformers torch
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('Dependencies installed.')

Dependencies installed.


In [2]:
# Cell 2: Imports
import json, csv, time, re, math, os
from dataclasses import dataclass, field
from typing import List, Dict, Tuple
import numpy as np
import torch
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from nltk.tokenize import sent_tokenize

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

Using device: cuda


In [3]:
# Cell 3: Load models (runs once, ~2 min)
print('Loading BGE-small...')
bi_encoder = SentenceTransformer('BAAI/bge-small-en-v1.5', device=DEVICE)
print('Loading cross-encoder reranker...')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=DEVICE)
print('Models loaded.')

Loading BGE-small...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading cross-encoder reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded.


In [4]:
# Cell 4: Chunking strategies

def tokenize_simple(text):
    return text.split()

def chunk_fixed(text, chunk_tokens=256, overlap_tokens=32):
    words = text.split()
    chunks = []
    step = chunk_tokens - overlap_tokens
    for i in range(0, len(words), step):
        chunk = ' '.join(words[i:i+chunk_tokens])
        if chunk.strip():
            chunks.append(chunk)
    return chunks

def chunk_sentence(text, soft_limit=300):
    sentences = sent_tokenize(text)
    chunks, current, current_len = [], [], 0
    for sent in sentences:
        sent_len = len(sent.split())
        if current_len + sent_len > soft_limit and current:
            chunks.append(' '.join(current))
            current, current_len = [], 0
        current.append(sent)
        current_len += sent_len
    if current:
        chunks.append(' '.join(current))
    return [c for c in chunks if c.strip()]

CHUNK_STRATEGIES = {
    'fixed_256': lambda t: chunk_fixed(t, 256, 32),
    'fixed_512': lambda t: chunk_fixed(t, 512, 64),
    'sent_300':  lambda t: chunk_sentence(t, 300),
}
print('Chunking strategies defined.')

Chunking strategies defined.


In [5]:
# Cell 5: Retrieval functions

def encode_chunks(chunks, model, batch_size=64):
    return model.encode(chunks, batch_size=batch_size,
                        normalize_embeddings=True,
                        show_progress_bar=False)

def bm25_retrieve(query, chunks, bm25_index, top_k=50):
    tokenized_q = query.split()
    scores = bm25_index.get_scores(tokenized_q)
    ranked = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in ranked]

def dense_retrieve(query, chunk_embeddings, model, top_k=50):
    q_emb = model.encode([query], normalize_embeddings=True)[0]
    scores = chunk_embeddings @ q_emb
    ranked = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in ranked]

def rrf_fuse(ranked_lists, k=60):
    scores = {}
    for ranked in ranked_lists:
        for rank, (idx, _) in enumerate(ranked):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def rerank(query, chunks, candidate_ids_scores, cross_enc, top_k=10):
    pairs = [(query, chunks[i]) for i, _ in candidate_ids_scores[:50]]
    ce_scores = cross_enc.predict(pairs)
    reranked = sorted(zip([i for i, _ in candidate_ids_scores[:50]], ce_scores),
                      key=lambda x: x[1], reverse=True)
    return reranked[:top_k]

print('Retrieval functions defined.')

Retrieval functions defined.


In [6]:
# Cell 6: Evaluation metrics

def recall_at_k(retrieved_ids, gold_ids, k):
    top_k = set([i for i, _ in retrieved_ids[:k]])
    return 1.0 if top_k & set(gold_ids) else 0.0

def mrr_at_k(retrieved_ids, gold_ids, k):
    gold_set = set(gold_ids)
    for rank, (i, _) in enumerate(retrieved_ids[:k], 1):
        if i in gold_set:
            return 1.0 / rank
    return 0.0

def avg(lst):
    return sum(lst) / len(lst) if lst else 0.0

print('Metrics defined.')

Metrics defined.


In [7]:
# Cell 7: Load QuALITY dataset
# Schema: article, question, options, answer (1-indexed int), hard (bool)
print('Loading QuALITY...')
quality_ds = load_dataset('emozilla/quality', split='validation')
print(f'QuALITY validation: {len(quality_ds)} examples')
print('Keys:', list(quality_ds[0].keys()))

Loading QuALITY...
QuALITY validation: 2086 examples
Keys: ['article', 'question', 'options', 'answer', 'hard']


In [8]:
# Cell 8: Prepare QuALITY
# Schema confirmed: article, question, options, answer (1-indexed), hard
# Each row = one question. Group by article text hash to build per-doc question lists.

from collections import defaultdict

def prepare_quality(ds, max_docs=100):
    doc_map = defaultdict(lambda: {'text': '', 'questions': []})

    for ex in ds:
        # Use hash of first 500 chars as stable doc_id (no article_id in schema)
        doc_id = str(hash(ex['article'][:500]))
        if not doc_map[doc_id]['text']:
            doc_map[doc_id]['text'] = ex['article']

        gold_idx = int(ex['answer']) - 1  # answer is 1-indexed
        options  = ex['options']
        if 0 <= gold_idx < len(options):
            doc_map[doc_id]['questions'].append({
                'question':    ex['question'],
                'gold_answer': options[gold_idx],
            })

    docs = []
    for doc_id, v in list(doc_map.items())[:max_docs]:
        text = v['text']
        if not text or not v['questions']:
            continue
        paragraphs = [p.strip() for p in text.split('\n') if p.strip()]

        qs = []
        for q_item in v['questions']:
            gold_answer = q_item['gold_answer']
            # Find paragraphs containing the gold answer text
            gold_para_indices = [
                i for i, p in enumerate(paragraphs)
                if gold_answer.lower()[:60] in p.lower()
            ]
            if not gold_para_indices:
                # Fallback: keyword overlap with answer words
                words = [w for w in gold_answer.split()[:6] if len(w) > 3]
                gold_para_indices = [
                    i for i, p in enumerate(paragraphs)
                    if any(w.lower() in p.lower() for w in words)
                ]
            if gold_para_indices:
                qs.append({
                    'question':          q_item['question'],
                    'gold_answer':       gold_answer,
                    'gold_para_indices': gold_para_indices,
                })

        if qs:
            docs.append({
                'doc_id':     doc_id,
                'text':       text,
                'paragraphs': paragraphs,
                'questions':  qs,
            })

    total_q = sum(len(d['questions']) for d in docs)
    print(f'Prepared {len(docs)} QuALITY documents, {total_q} questions')
    return docs

quality_docs = prepare_quality(quality_ds, max_docs=100)

Prepared 100 QuALITY documents, 1187 questions


In [9]:
# Cell 9: Load NarrativeQA
print('Loading NarrativeQA...')
nqa_ds = load_dataset('deepmind/narrativeqa', split='validation')
print(f'NarrativeQA validation: {len(nqa_ds)} examples')
print('Keys:', list(nqa_ds[0].keys()))

Loading NarrativeQA...


Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

data/train-00010-of-00024.parquet:   0%|          | 0.00/126M [00:00<?, ?B/s]

data/train-00016-of-00024.parquet:   0%|          | 0.00/194M [00:00<?, ?B/s]

...

data/validation-00002-of-00003.parquet:   0%|          | 0.00/68.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/32747 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10557 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3461 [00:00<?, ? examples/s]

NarrativeQA validation: 3461 examples
Keys: ['document', 'question', 'answers']


In [10]:
# Cell 10: Prepare NarrativeQA
def prepare_nqa(ds, max_docs=50):
    from collections import defaultdict
    doc_map = defaultdict(lambda: {'text': '', 'questions': []})
    for ex in ds:
        doc_id = ex['document']['id']
        if not doc_map[doc_id]['text']:
            doc_map[doc_id]['text'] = ex['document']['summary']['text']
        answers = [a['text'] for a in ex['answers']]
        doc_map[doc_id]['questions'].append({
            'question': ex['question']['text'],
            'answers': answers
        })
    docs = []
    for doc_id, v in list(doc_map.items())[:max_docs]:
        text = v['text']
        paragraphs = [p.strip() for p in text.split('\n') if p.strip()]
        qs = []
        for q_item in v['questions']:
            gold_paras = []
            for ans in q_item['answers']:
                for i, p in enumerate(paragraphs):
                    if ans.lower()[:30] in p.lower():
                        gold_paras.append(i)
                        break
            if gold_paras:
                qs.append({'question': q_item['question'],
                           'answers': q_item['answers'],
                           'gold_para_indices': list(set(gold_paras))})
        if qs:
            docs.append({'doc_id': doc_id, 'text': text,
                         'paragraphs': paragraphs, 'questions': qs})
    print(f'Prepared {len(docs)} NarrativeQA documents, '
          f'{sum(len(d["questions"]) for d in docs)} questions')
    return docs

nqa_docs = prepare_nqa(nqa_ds, max_docs=50)

Prepared 50 NarrativeQA documents, 785 questions


In [11]:
# Cell 11: Core evaluation loop
# This runs all 18 configurations (3 chunk x 3 retrieval x 2 rerank)
# on a given set of documents.
# Expected runtime: ~2 hrs per dataset on free T4

def run_experiment(docs, dataset_name, bi_enc, cross_enc):
    results = []
    total_configs = len(CHUNK_STRATEGIES) * 3 * 2  # 18
    config_num = 0

    for chunk_name, chunk_fn in CHUNK_STRATEGIES.items():
        print(f'\n=== Chunking: {chunk_name} ===')

        for doc in docs:
            doc['chunks_' + chunk_name] = chunk_fn(doc['text'])

        # Encode all chunks with BGE (do once per chunking strategy)
        print(f'  Encoding chunks with BGE-small...')
        all_chunks_flat = []
        doc_chunk_offsets = []  # (start_idx, end_idx) per doc
        for doc in docs:
            start = len(all_chunks_flat)
            all_chunks_flat.extend(doc['chunks_' + chunk_name])
            doc_chunk_offsets.append((start, len(all_chunks_flat)))

        all_embeddings = bi_enc.encode(
            all_chunks_flat, batch_size=64,
            normalize_embeddings=True, show_progress_bar=True
        )

        for retrieval_type in ['bm25', 'dense', 'hybrid']:
            for use_rerank in [False, True]:
                config_num += 1
                config_name = f'{chunk_name}_{retrieval_type}_{"rerank" if use_rerank else "norerank"}'
                print(f'  Config {config_num}/{total_configs}: {config_name}')

                r1_list, r5_list, r10_list, mrr_list = [], [], [], []
                chunk_count_list = []

                for doc_idx, doc in enumerate(docs):
                    chunks = doc['chunks_' + chunk_name]
                    if not chunks or not doc['questions']:
                        continue
                    chunk_count_list.append(len(chunks))

                    start, end = doc_chunk_offsets[doc_idx]
                    doc_embeddings = all_embeddings[start:end]

                    # Build BM25 for this doc
                    tokenized = [c.split() for c in chunks]
                    bm25 = BM25Okapi(tokenized)

                    # Find gold chunk indices for each question
                    # Gold: chunk that contains the most overlap with gold paragraph
                    paragraphs = doc['paragraphs']

                    for q_item in doc['questions']:
                        query = q_item['question']
                        gold_para_indices = q_item['gold_para_indices']
                        gold_texts = [paragraphs[i] for i in gold_para_indices
                                      if i < len(paragraphs)]
                        if not gold_texts:
                            continue

                        # Map gold paragraphs to chunk indices
                        gold_chunk_ids = set()
                        for gold_text in gold_texts:
                            gold_words = set(gold_text.lower().split())
                            best_overlap, best_idx = 0, 0
                            for ci, chunk in enumerate(chunks):
                                chunk_words = set(chunk.lower().split())
                                overlap = len(gold_words & chunk_words)
                                if overlap > best_overlap:
                                    best_overlap = overlap
                                    best_idx = ci
                            if best_overlap > 3:
                                gold_chunk_ids.add(best_idx)
                        if not gold_chunk_ids:
                            continue

                        # Retrieve
                        bm25_results = bm25_retrieve(query, chunks, bm25, top_k=50)
                        dense_results = dense_retrieve(query, doc_embeddings, bi_enc, top_k=50)

                        if retrieval_type == 'bm25':
                            candidates = bm25_results
                        elif retrieval_type == 'dense':
                            candidates = dense_results
                        else:  # hybrid
                            candidates = rrf_fuse([bm25_results, dense_results])

                        if use_rerank:
                            final_results = rerank(query, chunks, candidates, cross_enc, top_k=10)
                        else:
                            final_results = candidates[:10]

                        r1_list.append(recall_at_k(final_results, gold_chunk_ids, 1))
                        r5_list.append(recall_at_k(final_results, gold_chunk_ids, 5))
                        r10_list.append(recall_at_k(final_results, gold_chunk_ids, 10))
                        mrr_list.append(mrr_at_k(final_results, gold_chunk_ids, 10))

                if r1_list:
                    results.append({
                        'dataset': dataset_name,
                        'chunking': chunk_name,
                        'retrieval': retrieval_type,
                        'rerank': use_rerank,
                        'config': config_name,
                        'n_questions': len(r1_list),
                        'avg_chunks_per_doc': round(avg(chunk_count_list), 1),
                        'recall@1':  round(avg(r1_list),  4),
                        'recall@5':  round(avg(r5_list),  4),
                        'recall@10': round(avg(r10_list), 4),
                        'mrr@10':    round(avg(mrr_list), 4),
                    })
                    print(f'    R@10={avg(r10_list):.3f} MRR={avg(mrr_list):.3f} '
                          f'({len(r1_list)} questions)')
    return results

print('Experiment loop defined. Ready to run.')

Experiment loop defined. Ready to run.


In [12]:
# Cell 12: RUN QuALITY experiment (~2 hrs)
print('Starting QuALITY experiment...')
t0 = time.time()
quality_results = run_experiment(quality_docs, 'QuALITY', bi_encoder, cross_encoder)
print(f'\nQuALITY done in {(time.time()-t0)/60:.1f} min')

Starting QuALITY experiment...

=== Chunking: fixed_256 ===
  Encoding chunks with BGE-small...


Batches:   0%|          | 0/31 [00:00<?, ?it/s]

  Config 1/18: fixed_256_bm25_norerank
    R@10=0.938 MRR=0.663 (1184 questions)
  Config 2/18: fixed_256_bm25_rerank
    R@10=0.948 MRR=0.695 (1184 questions)
  Config 3/18: fixed_256_dense_norerank
    R@10=0.948 MRR=0.679 (1184 questions)
  Config 4/18: fixed_256_dense_rerank
    R@10=0.948 MRR=0.695 (1184 questions)
  Config 5/18: fixed_256_hybrid_norerank
    R@10=0.948 MRR=0.679 (1184 questions)
  Config 6/18: fixed_256_hybrid_rerank
    R@10=0.948 MRR=0.695 (1184 questions)

=== Chunking: fixed_512 ===
  Encoding chunks with BGE-small...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

  Config 7/18: fixed_512_bm25_norerank
    R@10=0.991 MRR=0.787 (1184 questions)
  Config 8/18: fixed_512_bm25_rerank
    R@10=0.992 MRR=0.791 (1184 questions)
  Config 9/18: fixed_512_dense_norerank
    R@10=0.993 MRR=0.778 (1184 questions)
  Config 10/18: fixed_512_dense_rerank
    R@10=0.992 MRR=0.791 (1184 questions)
  Config 11/18: fixed_512_hybrid_norerank
    R@10=0.991 MRR=0.790 (1184 questions)
  Config 12/18: fixed_512_hybrid_rerank
    R@10=0.992 MRR=0.791 (1184 questions)

=== Chunking: sent_300 ===
  Encoding chunks with BGE-small...


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

  Config 13/18: sent_300_bm25_norerank
    R@10=0.956 MRR=0.711 (1184 questions)
  Config 14/18: sent_300_bm25_rerank
    R@10=0.963 MRR=0.735 (1184 questions)
  Config 15/18: sent_300_dense_norerank
    R@10=0.964 MRR=0.726 (1184 questions)
  Config 16/18: sent_300_dense_rerank
    R@10=0.963 MRR=0.735 (1184 questions)
  Config 17/18: sent_300_hybrid_norerank
    R@10=0.965 MRR=0.726 (1184 questions)
  Config 18/18: sent_300_hybrid_rerank
    R@10=0.963 MRR=0.735 (1184 questions)

QuALITY done in 26.6 min


In [13]:
# Cell 13: RUN NarrativeQA experiment (~2 hrs)
print('Starting NarrativeQA experiment...')
t0 = time.time()
nqa_results = run_experiment(nqa_docs, 'NarrativeQA', bi_encoder, cross_encoder)
print(f'\nNarrativeQA done in {(time.time()-t0)/60:.1f} min')

Starting NarrativeQA experiment...

=== Chunking: fixed_256 ===
  Encoding chunks with BGE-small...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

  Config 1/18: fixed_256_bm25_norerank
    R@10=1.000 MRR=0.749 (785 questions)
  Config 2/18: fixed_256_bm25_rerank
    R@10=1.000 MRR=0.857 (785 questions)
  Config 3/18: fixed_256_dense_norerank
    R@10=1.000 MRR=0.787 (785 questions)
  Config 4/18: fixed_256_dense_rerank
    R@10=1.000 MRR=0.857 (785 questions)
  Config 5/18: fixed_256_hybrid_norerank
    R@10=1.000 MRR=0.763 (785 questions)
  Config 6/18: fixed_256_hybrid_rerank
    R@10=1.000 MRR=0.857 (785 questions)

=== Chunking: fixed_512 ===
  Encoding chunks with BGE-small...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

  Config 7/18: fixed_512_bm25_norerank
    R@10=1.000 MRR=0.786 (785 questions)
  Config 8/18: fixed_512_bm25_rerank
    R@10=1.000 MRR=0.941 (785 questions)
  Config 9/18: fixed_512_dense_norerank
    R@10=1.000 MRR=0.886 (785 questions)
  Config 10/18: fixed_512_dense_rerank
    R@10=1.000 MRR=0.941 (785 questions)
  Config 11/18: fixed_512_hybrid_norerank
    R@10=1.000 MRR=0.787 (785 questions)
  Config 12/18: fixed_512_hybrid_rerank
    R@10=1.000 MRR=0.941 (785 questions)

=== Chunking: sent_300 ===
  Encoding chunks with BGE-small...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

  Config 13/18: sent_300_bm25_norerank
    R@10=1.000 MRR=0.770 (785 questions)
  Config 14/18: sent_300_bm25_rerank
    R@10=1.000 MRR=0.893 (785 questions)
  Config 15/18: sent_300_dense_norerank
    R@10=1.000 MRR=0.836 (785 questions)
  Config 16/18: sent_300_dense_rerank
    R@10=1.000 MRR=0.893 (785 questions)
  Config 17/18: sent_300_hybrid_norerank
    R@10=1.000 MRR=0.779 (785 questions)
  Config 18/18: sent_300_hybrid_rerank
    R@10=1.000 MRR=0.893 (785 questions)

NarrativeQA done in 4.4 min


In [14]:
# Cell 14: Save results to CSV and display
import csv

all_results = quality_results + nqa_results

if all_results:
    keys = list(all_results[0].keys())
    with open('rag_experiment_results.csv', 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(all_results)
    print('Results saved to rag_experiment_results.csv')

# Pretty print
print('\n=== FINAL RESULTS ===')
print(f'{"Config":<50} {"R@1":>6} {"R@5":>6} {"R@10":>6} {"MRR":>6}')
print('-'*76)
for r in all_results:
    label = f"{r['dataset']} | {r['config']}"
    print(f'{label:<50} {r["recall@1"]:>6.3f} {r["recall@5"]:>6.3f} '
          f'{r["recall@10"]:>6.3f} {r["mrr@10"]:>6.3f}')

Results saved to rag_experiment_results.csv

=== FINAL RESULTS ===
Config                                                R@1    R@5   R@10    MRR
----------------------------------------------------------------------------
QuALITY | fixed_256_bm25_norerank                   0.520  0.855  0.938  0.663
QuALITY | fixed_256_bm25_rerank                     0.568  0.867  0.949  0.695
QuALITY | fixed_256_dense_norerank                  0.547  0.858  0.948  0.679
QuALITY | fixed_256_dense_rerank                    0.568  0.867  0.949  0.695
QuALITY | fixed_256_hybrid_norerank                 0.539  0.862  0.948  0.679
QuALITY | fixed_256_hybrid_rerank                   0.568  0.867  0.949  0.695
QuALITY | fixed_512_bm25_norerank                   0.677  0.936  0.991  0.787
QuALITY | fixed_512_bm25_rerank                     0.683  0.938  0.992  0.791
QuALITY | fixed_512_dense_norerank                  0.661  0.939  0.993  0.778
QuALITY | fixed_512_dense_rerank                    0.683  0.938  

In [15]:
# Cell 15: Download results
from google.colab import files
files.download('rag_experiment_results.csv')
print('Download started. Send the CSV file back to your immigration buddy.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started. Send the CSV file back to your immigration buddy.
